# Prompt Engineering & LCEL (LangChain Expression Language)

This notebook covers the foundations of enterprise Prompt Engineering and the **LangChain Expression Language (LCEL)**—the declarative, unified orchestration layer in modern LangChain.

### Key Learning Objectives:
1. Master Prompt Engineering principles: Role, Task, Context, Constraints, and Output Format.
2. Distinguish between Raw Strings, `PromptTemplate`, and `ChatPromptTemplate`.
3. Manage multi-turn conversational history with `MessagesPlaceholder`.
4. Compose modular pipelines using the LCEL pipe operator (`|`) and `StrOutputParser`.
5. Orchestrate advanced data flows with `RunnablePassthrough`, `RunnableParallel`, and `RunnableLambda`.

## 1. Environment Setup

We initialize our environment and connect to Google Gemini (or any configured LLM).

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)
print("Model initialized:", model.model)

## 2. Prompt Templates: No-Template vs PromptTemplate vs ChatPromptTemplate

1. **No-Template (Raw Strings)**: Passing raw hardcoded strings directly to `model.invoke()`. Prone to manual formatting bugs and difficult to reuse.
2. **`PromptTemplate`**: Used primarily for text completion models or single string formatting with parameterized `{variable}` slots.
3. **`ChatPromptTemplate`**: Designed for chat models. Represents conversations as structured sequences of roles (`system`, `human`, `ai`).

In [ ]:
# 1. No-template: Direct raw string invocation
raw_response = model.invoke("Explain ACID properties in relational databases in one sentence.")
print("--- 1. Raw String Output ---")
print(raw_response.content)

# 2. PromptTemplate: String-level parameterization
string_prompt = PromptTemplate.from_template(
    "Explain the concept of {concept} to a {target_audience} in {num_bullet_points} bullet points."
)
formatted_str = string_prompt.format(
    concept="Decentralized Consensus",
    target_audience="high school student",
    num_bullet_points=3
)
print("\n--- 2. Formatted PromptTemplate ---")
print(formatted_str)

# 3. ChatPromptTemplate: Role-structured conversation schema
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite software architect specializing in {domain}. Always respond with clear technical justifications."),
    ("human", "What are the trade-offs of using {architecture_pattern}?")
])

formatted_chat = chat_prompt.format_messages(
    domain="high-throughput microservices",
    architecture_pattern="Event Sourcing with CQRS"
)
chat_response = model.invoke(formatted_chat)
print("\n--- 3. ChatPromptTemplate Output ---")
print(chat_response.content[:300] + "...\n[Truncated]")

## 3. Dynamic History Injection with MessagesPlaceholder

`MessagesPlaceholder` allows you to dynamically inject a variable list of prior conversation messages (such as `HumanMessage`, `AIMessage`, or `SystemMessage`) into a fixed prompt template.

This is critical for conversational memory and multi-turn stateful chatbots.

In [ ]:
# ChatPromptTemplate with MessagesPlaceholder for history injection
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful mathematical tutor. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_question}")
])

# Simulating an existing multi-turn conversation
history = [
    HumanMessage(content="Let x = 15."),
    AIMessage(content="Got it. x is equal to 15."),
    HumanMessage(content="Let y = 2 * x + 5."),
    AIMessage(content="Understood. Calculating y: y = 2(15) + 5 = 35.")
]

# Formatting prompt with history and a new question referencing prior state
final_messages = prompt_with_history.format_messages(
    chat_history=history,
    user_question="Now calculate x + y and divide the result by 5."
)

response_math = model.invoke(final_messages)
print("Conversation with MessagesPlaceholder Response:")
print(response_math.content)

## 4. LCEL Core Foundations: The Pipe Operator (`|`) & StrOutputParser

**LangChain Expression Language (LCEL)** allows you to compose components into execution pipelines using the Python bitwise OR / pipe operator (`|`).

Every LCEL component implements the **Runnable protocol**, exposing standard methods:
* `.invoke(input)`: Run synchronously on a single input.
* `.stream(input)`: Stream output chunks in real-time.
* `.batch([input1, input2])`: Run in parallel across multiple inputs.
* `.ainvoke(input)`: Async execution.

`StrOutputParser()` automatically extracts the raw string content from the model's `AIMessage` object, eliminating repetitive `.content` property access.

In [ ]:
# Composing a foundational LCEL chain
prompt = ChatPromptTemplate.from_template(
    "Explain the concept of {technology} in 2 concise sentences for a junior developer."
)
parser = StrOutputParser()

# LCEL Pipeline: Prompt -> Model -> StrOutputParser
chain = prompt | model | parser

# Invoking the chain
result = chain.invoke({"technology": "Docker containers"})
print("--- LCEL String Parsed Output ---")
print(result)

# Streaming directly from the chain
print("\n--- Streaming LCEL Chain ---")
for chunk in chain.stream({"technology": "Redis In-Memory Cache"}):
    print(chunk, end="", flush=True)
print()

## 5. Advanced LCEL: RunnablePassthrough, RunnableParallel & RunnableLambda

* **`RunnablePassthrough`**: Passes input through unchanged, or allows adding extra keys to a dictionary while keeping existing keys.
* **`RunnableParallel`** (or dict syntax `{key1: ..., key2: ...}`): Executes multiple runnable branches concurrently on the same input.
* **`RunnableLambda`**: Converts any custom Python function or lambda into a first-class Runnable component in the chain.

In [ ]:
# 1. Custom transformation function using RunnableLambda
def word_count(text: str) -> dict:
    words = len(text.split())
    chars = len(text)
    return {"word_count": words, "character_count": chars}

lambda_runner = RunnableLambda(word_count)

# 2. Parallel execution branches with RunnableParallel
pros_prompt = ChatPromptTemplate.from_template("List 2 major advantages of {topic}.")
cons_prompt = ChatPromptTemplate.from_template("List 2 major disadvantages or risks of {topic}.")

pros_chain = pros_prompt | model | StrOutputParser()
cons_chain = cons_prompt | model | StrOutputParser()

parallel_chain = RunnableParallel(
    topic=RunnablePassthrough(),
    advantages=pros_chain,
    risks=cons_chain
)

analysis = parallel_chain.invoke("Serverless Cloud Architecture")
print("--- Parallel Branch Execution Results ---")
print("TOPIC:", analysis["topic"])
print("\n[ADVANTAGES]:\n", analysis["advantages"])
print("\n[RISKS]:\n", analysis["risks"])

## 6. End-to-End Multi-Step Pipeline (Technical Digest & Translation)

Here we compose a multi-stage sequential pipeline where:
1. Stage 1 summarizes technical documentation into bullet points.
2. Stage 2 takes the summary and translates it into French for an international team.
3. The chain chains both stages smoothly via LCEL piping.

In [ ]:
# Stage 1: Technical Summarization Prompt
summary_prompt = ChatPromptTemplate.from_template(
    "Synthesize the following engineering text into 3 core technical takeaways:\n\n{text}"
)

# Stage 2: Multilingual Translation Prompt
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following takeaways into {target_language}. Maintain technical precision:\n\n{summary}"
)

# Multi-stage LCEL Chain
summarize_chain = summary_prompt | model | StrOutputParser()

full_pipeline = (
    RunnableParallel(
        summary=summarize_chain,
        target_language=lambda x: x["target_language"]
    )
    | translate_prompt
    | model
    | StrOutputParser()
)

sample_text = """
Kubernetes is an open-source system for automating deployment, scaling, and management of containerized applications.
It groups containers that make up an application into logical units for easy management and discovery.
Kubernetes builds upon 15 years of experience of running production workloads at Google, combined with best-of-breed ideas and practices from the community.
It provides service discovery, load balancing, automated rollouts, rollback mechanisms, and self-healing.
"""

final_translation = full_pipeline.invoke({
    "text": sample_text,
    "target_language": "French"
})

print("--- End-to-End Pipeline Output (French Summary) ---")
print(final_translation)